# Choosing a Fusion Method: RRF, DBSF, and FormulaQuery for Hybrid Search

| Time: 45 min | Level: Intermediate | Requires: Qdrant 1.17.x or later |
| --- | --- | --- |

Hybrid search combines two or more retrievers (typically dense + sparse) into a single ranking. Qdrant ships **RRF** and **DBSF** for fusion, plus **`FormulaQuery`** for ranking logic on top (recency, boosts, geo). This notebook compares the three on [BEIR/SciFact](https://huggingface.co/datasets/BeIR/scifact) (5,183 documents, 300 queries with relevance labels) and walks through tuning weighted RRF when you have an eval set.

Everything runs against a [Qdrant Cloud Free Tier Cluster](https://qdrant.tech/documentation/cloud/create-cluster/#free-clusters) with [Cloud Inference](https://qdrant.tech/documentation/cloud/inference/) for both dense and BM25 embeddings, so no local model downloads are required.

## Setup

In [ ]:
!pip install -q "qdrant-client>=1.17.0" datasets ranx pandas numpy

<aside role="status">This notebook uses <a href="/documentation/inference/#qdrant-cloud-inference">Qdrant Cloud Inference</a> to generate embeddings server-side. The free tier covers this footprint. Core BM25 runs on any Qdrant instance, but dense Cloud Inference is Cloud-only. To self-host, generate dense vectors on the client with <a href="/documentation/fastembed/">FastEmbed</a> and pass them as raw vectors instead of <code>models.Document</code>.</aside>

In [ ]:
import random
from collections import defaultdict

import numpy as np
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
from qdrant_client import QdrantClient, models
from ranx import Qrels, Run, evaluate

# Set QDRANT_URL and QDRANT_API_KEY as Colab secrets (left sidebar, key icon).
# Get the values from https://cloud.qdrant.io
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    cloud_inference=True,
)

COLLECTION = "scifact_fusion"
DENSE_MODEL = "sentence-transformers/all-minilm-l6-v2"
BM25_MODEL = "qdrant/bm25"
DENSE_DIM = 384
BM25_AVG_LEN = 220  # SciFact abstracts average roughly 220 words; tune to your corpus

random.seed(42)
np.random.seed(42)

## Load BEIR/SciFact

SciFact is a small fact-checking corpus. We use the BEIR `test` split, which provides 300 queries each paired with their known-relevant documents.

In [ ]:
corpus_ds = load_dataset("BeIR/scifact", "corpus", split="corpus")
queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

point_to_id = {i: doc["_id"] for i, doc in enumerate(corpus_ds)}

query_ids_with_qrels = {str(row["query-id"]) for row in qrels_ds}
eval_queries = [q for q in queries_ds if q["_id"] in query_ids_with_qrels]

qrels_dict = defaultdict(dict)
for row in qrels_ds:
    qrels_dict[str(row["query-id"])][str(row["corpus-id"])] = int(row["score"])

print(f"Corpus: {len(corpus_ds)} documents")
print(f"Eval queries: {len(eval_queries)}")
print(f"Known-relevant (query, doc) pairs: {sum(len(v) for v in qrels_dict.values())}")

## Create the Collection

Two retrievers stored as named vectors on the same point:

- **`dense`**: `all-minilm-l6-v2` (384 dimensions, cosine).
- **`bm25`**: sparse, with `Modifier.IDF` so Qdrant computes inverse document frequency from the indexed corpus. Without this modifier, scores would be term-frequency-only rather than full BM25.

In [ ]:
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

## Index the Corpus

We send raw text in both vector slots and let Cloud Inference produce the embeddings server-side.

In [ ]:
points = []
for i, doc in enumerate(corpus_ds):
    text = ((doc.get("title") or "") + " " + (doc.get("text") or "")).strip()
    points.append(
        models.PointStruct(
            id=i,
            vector={
                "dense": models.Document(text=text, model=DENSE_MODEL),
                "bm25": models.Document(
                    text=text,
                    model=BM25_MODEL,
                    # avg_len defaults to 256 (document-length). SciFact abstracts are
                    # ~220 words, so we tune it to the field's actual average. Important
                    # for BM25 ranking quality on short fields.
                    options={"avg_len": BM25_AVG_LEN},
                ),
            },
            payload={"doc_id": doc["_id"], "title": doc.get("title", ""), "text": doc.get("text", "")},
        )
    )

client.upload_points(
    collection_name=COLLECTION,
    points=points,
    batch_size=256,
)
print(f"Indexed {len(points)} points")

## 1. Combining Two Ranked Lists

Hybrid search returns two ranked lists per query. The first instinct is to combine them with a tunable weight like `alpha * dense + (1 - alpha) * sparse`. Let's look at the raw scores for one query to see how that combination behaves in practice.

In [21]:
demo_query = eval_queries[0]["text"]
print(f"Query: {demo_query!r}\n")

dense_hits = client.query_points(
    collection_name=COLLECTION,
    query=models.Document(text=demo_query, model=DENSE_MODEL),
    using="dense",
    limit=20,
    with_payload=False,
).points

sparse_hits = client.query_points(
    collection_name=COLLECTION,
    query=models.Document(text=demo_query, model=BM25_MODEL, options={"avg_len": BM25_AVG_LEN}),
    using="bm25",
    limit=20,
    with_payload=False,
).points

print("Dense top scores:", [round(h.score, 3) for h in dense_hits[:10]])
print("BM25 top scores: ", [round(h.score, 3) for h in sparse_hits[:10]])

Query: '0-dimensional biomaterials show inductive properties.'

Dense top scores: [0.354, 0.331, 0.292, 0.291, 0.29, 0.282, 0.279, 0.271, 0.266, 0.26]
BM25 top scores:  [10.949, 10.814, 10.734, 10.469, 10.233, 10.138, 9.878, 9.643, 9.233, 9.076]


The dense scores are bounded (cosine, around 0.3 to 0.7) and the BM25 scores are unbounded positives (anywhere from 2 to 20+ depending on rare-term content). If you do `0.5 * dense + 0.5 * sparse`, BM25 dominates by an order of magnitude. Push toward `alpha = 0` or `alpha = 1` and you turn off one retriever entirely, which isn't fusion but selection. The BM25 scale also shifts per query, so a fixed alpha that performs well on one query distribution may not transfer cleanly to another without first normalizing the scores.

RRF and DBSF take different routes out of this. `FormulaQuery` sits in a different category (a ranking-logic layer on top of fusion); we'll cover it later in the FormulaQuery section.

## 2. Reciprocal Rank Fusion (RRF)

RRF sidesteps the score-scale problem by discarding scores and using only ranks. Each document's score becomes:

$$ \text{rrf}(d) = \sum_i \frac{1}{k + r_i(d)} $$

where $r_i(d)$ is the document's rank in retriever $i$ and $k$ is a smoothing constant (default 2 in Qdrant; classic literature uses 60). Ranks are on the same scale by construction, so combining them is well-defined.

**RRF: things to know**

- Ranks ignore score magnitudes: a doc that's 10x better than its neighbor by raw score ranks the same as one that barely edges it out. In practice the loss is small, which is why RRF works well as a default.
- `k` defaults to 2 in Qdrant, but is tunable via `models.RrfQuery(rrf=models.Rrf(k=...))` (since v1.16). Smaller `k` sharpens the rank-1 advantage; larger `k` (the classic literature default of 60) smooths it.
- An eval set is optional with the default `k=2`, but worth having if you change `k`.

Before the first metric, a brief note on the eval setup: every method in this notebook is evaluated on the same 300 SciFact queries, so the side-by-side table is directly comparable. For notebook simplicity we tune and report on the same queries; in production you would hold out a val split. We flag the best-practice version again in the next section when we tune weighted-RRF weights.

In [41]:
TOP_K = 10
PREFETCH_LIMIT = 100

eval_qrels = Qrels({q["_id"]: qrels_dict[q["_id"]] for q in eval_queries if q["_id"] in qrels_dict})

print(f"Eval queries: {len(eval_queries)}")

METRICS = ["ndcg@10", "recall@100", "mrr@10"]


def run_fusion(query_text, fusion_query, top_k=TOP_K):
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL),
                using="dense",
                limit=PREFETCH_LIMIT,
            ),
            models.Prefetch(
                query=models.Document(text=query_text, model=BM25_MODEL, options={"avg_len": BM25_AVG_LEN}),
                using="bm25",
                limit=PREFETCH_LIMIT,
            ),
        ],
        query=fusion_query,
        limit=top_k,
        with_payload=["doc_id"],
    ).points


def run_single(query_text, using, top_k=TOP_K):
    if using == "dense":
        query = models.Document(text=query_text, model=DENSE_MODEL)
    else:
        query = models.Document(text=query_text, model=BM25_MODEL, options={"avg_len": BM25_AVG_LEN})
    return client.query_points(
        collection_name=COLLECTION,
        query=query,
        using=using,
        limit=top_k,
        with_payload=["doc_id"],
    ).points


def build_run(name, query_fn):
    """Build a ranx Run over the eval queries."""
    run_dict = {}
    for q in eval_queries:
        hits = query_fn(q["text"])
        run_dict[q["_id"]] = {point_to_id[h.id]: float(h.score) for h in hits}
    return Run(run_dict, name=name)


results_table = {}

Eval queries: 300


In [42]:
dense_only = build_run("dense", lambda q: run_single(q, "dense", top_k=100))
sparse_only = build_run("bm25", lambda q: run_single(q, "bm25", top_k=100))

rrf_run = build_run(
    "rrf",
    lambda q: run_fusion(q, models.RrfQuery(rrf=models.Rrf()), top_k=100),
)

for run in (dense_only, sparse_only, rrf_run):
    results_table[run.name] = {m: evaluate(eval_qrels, run, m) for m in METRICS}

pd.DataFrame(results_table).T

,ndcg@10,recall@100,mrr@10
dense,0.654084,0.931667,0.607063
bm25,0.683446,0.924667,0.647511
rrf,0.723467,0.958333,0.681292


**What these metrics measure**

- **nDCG@10**: rewards relevant docs near the top with logarithmic position discount. The most informative single number for ranking quality.
- **Recall@100**: did the prefetch find every relevant doc? Ranking doesn't matter, only inclusion. A reranker downstream can fix top-10 order, but it can't recover docs lost to the prefetch ceiling.
- **MRR@10**: position of the first relevant doc. Best for navigational or single-answer tasks where the top result is what counts.

**Reference numbers on SciFact:**

| Method | nDCG@10 | Recall@100 | MRR@10 |
| --- | --- | --- | --- |
| dense | 0.654 | 0.932 | 0.607 |
| bm25  | 0.683 | 0.925 | 0.648 |
| rrf   | 0.723 | 0.958 | 0.681 |

RRF beats both single-retriever baselines on every metric: +0.04 nDCG@10 over BM25, +0.03 Recall@100, +0.03 MRR@10. If your row lands materially below either baseline, the usual suspects are a too-small `PREFETCH_LIMIT`, an `avg_len` mismatch on BM25, or low diversity between retrievers (both finding the same docs).

## 3. Weighted RRF and How to Pick Weights

Default RRF treats both retrievers equally. In real applications one retriever is usually stronger (often dense on natural language, BM25 on identifier-heavy queries). Qdrant's `RrfQuery` takes a `weights` array. The question is how to pick the numbers.

A grid search is a simple and common approach. Here's a reusable helper that searches over a list of weight tuples and scores each on the eval set. For notebook simplicity we tune and report on the same queries; we'll flag the best-practice change for production data after the sweep.

**Weighted RRF: when to consider it**

- Helpful when you have an eval set (queries paired with known-relevant docs) and the retrievers differ in strength. With a small set (under ~50 queries) or matched retrievers, the default `(1.0, 1.0)` is usually fine.
- Worth retuning when retrievers, corpus, or chunking change substantially.

In [43]:
def tune_rrf_weights(
    eval_queries,
    qrels_dict,
    fusion_runner,
    weight_grid,
    metric="ndcg@10",
):
    """Grid-search weighted RRF weights on the eval set.

    Note: this tunes and reports on the same queries for notebook simplicity.
    In production, hold out a val split: pick weights on train, report metrics
    on val. See the best-practice callout after the sweep table.

    fusion_runner: callable (query_text, weights) -> list of hits with .id and .score
    weight_grid:   list of weight tuples, e.g. [(1.0, 1.0), (2.0, 1.0), (1.0, 2.0)]
    Returns: dict with best_weights, sweep_table (DataFrame), best_score.
    """
    eval_qrels = Qrels({q["_id"]: qrels_dict[q["_id"]] for q in eval_queries if q["_id"] in qrels_dict})

    rows = []
    for weights in weight_grid:
        run = {q["_id"]: {point_to_id[h.id]: float(h.score) for h in fusion_runner(q["text"], weights)} for q in eval_queries}
        score = evaluate(eval_qrels, Run(run, name=f"w={weights}"), metric)
        rows.append({"weights": weights, metric: score})

    sweep = pd.DataFrame(rows).sort_values(metric, ascending=False).reset_index(drop=True)
    best = sweep.iloc[0]
    return {
        "best_weights": best["weights"],
        "sweep_table": sweep,
        "best_score": best[metric],
    }

### Running the grid search

The next cell calls `tune_rrf_weights` with 8 weight tuples and scores each on the full eval set, then reports the best weights and their nDCG@10.

> **Heads up: this cell takes ~10-30 minutes on Cloud Inference free tier.** The grid search fires roughly 2,400 `query_points` calls (8 weight tuples × 300 queries, each generating two server-side embeddings).
>
> If you haven't changed anything earlier in the notebook, the saved outputs from a previous run are already populated. You can read along without rerunning.

In [44]:
def run_weighted_rrf(query_text, weights, top_k=100):
    return run_fusion(
        query_text,
        models.RrfQuery(rrf=models.Rrf(weights=list(weights))),
        top_k=top_k,
    )

weight_grid = [
    (1.0, 3.0), (1.0, 2.0), (1.0, 1.5), (1.0, 1.0),
    (1.5, 1.0), (2.0, 1.0), (3.0, 1.0), (5.0, 1.0),
]

tuning = tune_rrf_weights(
    eval_queries=eval_queries,
    qrels_dict=qrels_dict,
    fusion_runner=run_weighted_rrf,
    weight_grid=weight_grid,
)

print(f"Best weights (dense, bm25): {tuning['best_weights']}")
print(f"Best nDCG@10 on eval set:   {tuning['best_score']:.4f}")
tuning["sweep_table"]

Best weights (dense, bm25): (1.0, 2.0)
Best nDCG@10 on eval set:   0.7264


,weights,ndcg@10
0,"(1.0, 2.0)",0.726411
1,"(1.0, 1.0)",0.721844
2,"(1.0, 3.0)",0.721009
3,"(1.0, 1.5)",0.720400
4,"(1.5, 1.0)",0.717578
5,"(2.0, 1.0)",0.713063
6,"(5.0, 1.0)",0.706649
7,"(3.0, 1.0)",0.703982


On this SciFact run, the search picked `(1.0, 2.0)` as the best weights, with nDCG@10 = 0.7264. The default `(1.0, 1.0)` scored 0.7218, a 0.0046 gap. Every weight tuple in the grid landed within 0.025 nDCG of the top, so the metric surface is fairly flat on this dataset and retriever pair.

**A note on noise.** When we lock in `(1.0, 2.0)` and re-evaluate it in the next cell, the reported score will likely land closer to 0.721 than the 0.726 the sweep saw. Qdrant's HNSW retrieval is approximate. The gap between the sweep result and the re-evaluation will be roughly 0.005 nDCG, the same scale as the tuning lift itself. Trust the loop to surface a reasonable weight pair; don't read too much into small differences in the best-score number.

**Tuning typically helps more when retrievers differ in strength** (dense on natural language, BM25 on identifier-heavy queries). On SciFact with this retriever pair, dense and BM25 are well-matched, so tuning here is effectively a no-op.

For production, retune when your retrievers change (new model, new chunking), when your corpus drifts substantially, or roughly every quarter on a fresh eval sample.

> **Best practice for real data: hold out a val split.** This notebook tunes and reports on the same set of queries for simplicity. In production:
>
> - Split your queries into train (~70%) and val (~30%) before tuning.
> - Search the weight space on the **train** queries (these decide the winner).
> - Report metrics from the **held-out val** queries (these are what you can claim).
>
> Reporting on the same queries you tune on inflates the metric, because you literally picked the weights that maximize that score on those queries.

Now lock in the best weights and add the row to the comparison table:

In [45]:
best_w = tuning["best_weights"]
weighted_rrf_run = build_run(
    f"weighted_rrf{best_w}",
    lambda q: run_weighted_rrf(q, best_w, top_k=100),
)
results_table[weighted_rrf_run.name] = {m: evaluate(eval_qrels, weighted_rrf_run, m) for m in METRICS}
pd.DataFrame(results_table).T

,ndcg@10,recall@100,mrr@10
dense,0.654084,0.931667,0.607063
bm25,0.683446,0.924667,0.647511
rrf,0.723467,0.958333,0.681292
"weighted_rrf(1.0, 2.0)",0.721388,0.950333,0.679183


## 4. Distribution-Based Score Fusion (DBSF)

DBSF keeps the raw scores but normalizes their distributions before combining. For each retriever's returned set, Qdrant computes the mean $\mu$ and sample standard deviation $\sigma$, then linearly remaps every score using the 3-sigma extremes as endpoints:

$$ \hat{s} = \frac{s - (\mu - 3\sigma)}{6\sigma} $$

Normalized scores are summed across retrievers. Different score magnitudes no longer drive the combination because each retriever contributes on the same comparable range.

**Edge cases:** scores are **not** clipped to [0, 1]; values outside the 3-sigma range remain outside it. If all returned scores are identical (or only one point is returned), DBSF emits `0.5` rather than dividing by zero.

**DBSF: when to consider it**

- Works well when you trust your retrievers' raw score magnitudes to carry signal (well-calibrated dense + BM25 with corpus IDF is typical). Less reliable on retrievers with heavy-tailed score distributions.
- Normalization uses the prefetch top-k as its sample, so a small `PREFETCH_LIMIT` or a query with a dominant outlier can produce unstable rankings.
- No hyperparameters to tune; an eval set helps confirm whether DBSF outperforms RRF on your corpus.

Run DBSF on the eval set and add the row to the comparison table:

In [46]:
dbsf_run = build_run(
    "dbsf",
    lambda q: run_fusion(q, models.FusionQuery(fusion=models.Fusion.DBSF), top_k=100),
)
results_table["dbsf"] = {m: evaluate(eval_qrels, dbsf_run, m) for m in METRICS}
pd.DataFrame(results_table).T

,ndcg@10,recall@100,mrr@10
dense,0.654084,0.931667,0.607063
bm25,0.683446,0.924667,0.647511
rrf,0.723467,0.958333,0.681292
"weighted_rrf(1.0, 2.0)",0.721388,0.950333,0.679183
dbsf,0.736025,0.958333,0.702729


DBSF beat RRF on this run by +0.013 nDCG@10 (0.736 vs 0.723) and +0.022 MRR@10 (0.703 vs 0.681). Recall@100 ties at 0.958 across DBSF and plain RRF; weighted RRF trails marginally at 0.950.

## 5. Side-by-Side

All methods on the same metrics, sorted by nDCG@10:

In [47]:
summary = pd.DataFrame(results_table).T.sort_values("ndcg@10", ascending=False)
summary

,ndcg@10,recall@100,mrr@10
dbsf,0.736025,0.958333,0.702729
rrf,0.723467,0.958333,0.681292
"weighted_rrf(1.0, 2.0)",0.721388,0.950333,0.679183
bm25,0.683446,0.924667,0.647511
dense,0.654084,0.931667,0.607063


On this SciFact run, DBSF leads at 0.736 nDCG@10, RRF follows at 0.723, weighted RRF at the best `(1.0, 2.0)` lands at 0.721, then BM25 and dense bring up the rear. Recall@100 ties at 0.958 for DBSF and RRF, with weighted RRF marginally behind at 0.950. DBSF was highest on this run for both ranking metrics (nDCG and MRR).

The right method for your application depends on whether your retrievers' score magnitudes carry signal (DBSF), whether you have an eval set to tune on (weighted RRF), or whether you want a strong default (RRF). On SciFact with these retrievers, DBSF was the win. On a different corpus or retriever pair the order can flip, so re-run this notebook on your own data before committing.

For a deeper breakdown of when to prefer each, see the [FAQ on RRF vs. DBSF](/documentation/faq/qdrant-fundamentals/#when-should-i-use-reciprocal-rank-fusion-rrf-vs-distribution-based-score-fusion-dbsf-for-hybrid-search).

## 6. `FormulaQuery`: Fusion Plus Business Logic

`FormulaQuery` layers ranking logic on top of a fused result. The pattern is:

- Inner prefetch fuses dense + sparse with RRF (or DBSF).
- Outer `FormulaQuery` applies recency decay, popularity boosts, or category multipliers using the fused `$score` and payload fields.

It's not a way to write tuned-alpha fusion. Writing `0.7 * $score[0] + 0.3 * $score[1]` over raw retriever scores reintroduces the same scale problem that breaks naive linear fusion. If the prefetches are themselves `rrf` or `dbsf`, the scores are already on comparable scales and a weighted formula sum works.

**FormulaQuery: when to consider it**

- Decay coefficients need to be calibrated against the fused score scale. Small RRF scores plus an unweighted decay term that returns `[0, 1]` will let recency crowd out relevance.
- Multi-knob tuning is brittle without an eval set; single well-understood adjustments are easier to defend.
- Higher per-query latency than plain fusion because the engine evaluates the expression for each candidate.

Let's demonstrate recency decay. SciFact has no timestamps, so we'll inject a synthetic `published_at` per document. This section teaches the mechanics, not a metric.

In [48]:
from datetime import datetime, timedelta

rng = random.Random(0)
now = datetime(2026, 5, 14)

operations = []
for i in range(len(corpus_ds)):
    days_ago = rng.randint(0, 365 * 3)
    operations.append(
        models.SetPayloadOperation(
            set_payload=models.SetPayload(
                points=[i],
                payload={"published_at": (now - timedelta(days=days_ago)).isoformat() + "Z"},
            )
        )
    )

for start in range(0, len(operations), 500):
    client.batch_update_points(collection_name=COLLECTION, update_operations=operations[start:start + 500])

client.create_payload_index(
    collection_name=COLLECTION,
    field_name="published_at",
    field_schema=models.PayloadSchemaType.DATETIME,
)

UpdateResult(operation_id=20761, status=<UpdateStatus.COMPLETED: 'completed'>)

In [49]:
demo_query = eval_queries[0]["text"]

fused_only = run_fusion(demo_query, models.RrfQuery(rrf=models.Rrf()), top_k=5)

fused_then_decay = client.query_points(
    collection_name=COLLECTION,
    prefetch=models.Prefetch(
        prefetch=[
            models.Prefetch(
                query=models.Document(text=demo_query, model=DENSE_MODEL),
                using="dense",
                limit=PREFETCH_LIMIT,
            ),
            models.Prefetch(
                query=models.Document(text=demo_query, model=BM25_MODEL, options={"avg_len": BM25_AVG_LEN}),
                using="bm25",
                limit=PREFETCH_LIMIT,
            ),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=PREFETCH_LIMIT,
    ),
    query=models.FormulaQuery(
        formula=models.SumExpression(
            sum=[
                models.MultExpression(mult=[1.0, "$score"]),
                # Decay weight 0.1: RRF scores in the top-5 are roughly 0.2 to 0.5, while
                # un-weighted decay returns [0, 1]. Without this coefficient the decay
                # dominates the sum and recency replaces relevance instead of nudging it.
                models.MultExpression(
                    mult=[
                        0.1,
                        models.ExpDecayExpression(
                            exp_decay=models.DecayParamsExpression(
                                x=models.DatetimeKeyExpression(datetime_key="published_at"),
                                target=models.DatetimeExpression(datetime=now.isoformat() + "Z"),
                                scale=86400 * 180,
                                midpoint=0.5,
                            )
                        ),
                    ]
                ),
            ]
        )
    ),
    limit=5,
    with_payload=["doc_id", "published_at", "title"],
).points

print("--- RRF only ---")
for h in fused_only:
    print(f"  {h.id}  score={h.score:.4f}  {h.payload.get('title', '')[:70]}")

print("\n--- RRF + 0.1 * exp_decay on published_at (scale=180d, midpoint=0.5) ---")
for h in fused_then_decay:
    print(f"  {h.id}  score={h.score:.4f}  {h.payload.get('published_at')}  {h.payload.get('title', '')[:50]}")

--- RRF only ---
  4132  score=0.5000  
  4399  score=0.5000  
  766  score=0.3333  
  2891  score=0.3333  
  4241  score=0.2500  

--- RRF + 0.1 * exp_decay on published_at (scale=180d, midpoint=0.5) ---
  4399  score=0.5209  2025-04-02T00:00:00Z  The first gene of the Bacillus subtilis clpC opero
  4132  score=0.5153  2025-01-12T00:00:00Z  Complex Tissue and Disease Modeling using hiPSCs.
  766  score=0.3395  2024-05-21T00:00:00Z  Nonlinear Elasticity in Biological Gels
  2891  score=0.3374  2024-02-05T00:00:00Z  The Epithelial-Mesenchymal Transition Generates Ce
  4241  score=0.2976  2025-11-02T00:00:00Z  New opportunities: the use of nanotechnologies to 


The decay term tilts the ranking toward more recent documents. The `scale` and `midpoint` knobs control how aggressive the decay is: `scale=180 days, midpoint=0.5` means a doc 180 days from the target receives a decay value of 0.5, vs 1.0 for a brand-new doc.

The decay is wrapped in `MultExpression(mult=[0.1, ...])` because RRF scores are small (typically 0.2 to 0.5 in the top-5) while un-weighted decay returns `[0, 1]`. The `0.1` coefficient caps the decay's contribution so it nudges the ranking rather than replacing it. See [Search Relevance](/documentation/search/search-relevance/) for the full decay function reference.

Other patterns that fit `FormulaQuery` naturally:

- **Category boost:** multiply `$score` by 1.3 when `payload.category == "featured"`.
- **Popularity prior:** sum `$score` with `log(view_count + 1) / 10`.
- **Geo decay:** [`GaussDecayExpression`](/documentation/search/search-relevance/) on distance from a user's location.

## Picking a Method: At a Glance

| Factor | RRF | Weighted RRF | DBSF | FormulaQuery |
| --- | --- | --- | --- | --- |
| Eval set required | No | Yes | Recommended | Yes for multi-knob tuning |
| Hyperparameters to tune | `k` (optional, since v1.16) | Per-retriever weights, `k` (optional) | None | Decay scale, midpoint, term coefficients |
| Preserves score magnitudes | No | No | Yes | Depends on prefetch |
| Layers business logic | No | No | No | Yes |
| Fit for well-calibrated retrievers | OK | OK | **Strong** | OK (over fused prefetch) |
| Fit for retrievers with very different score scales | OK | OK with tuning | Limited | Limited (without normalized prefetches) |
| Latency overhead | None | None | Small | Medium |

## Wrap-up

- **RRF** is a strong default; ranks sidestep the score-scale problem.
- **Weighted RRF** is worth trying if you have an eval set. Tuned lifts are often within noise on well-matched retrievers (as on SciFact here).
- **DBSF** can outperform tuned RRF on well-calibrated retrievers; neither dominates the other in general.
- **`FormulaQuery`** layers business logic (recency, boosts, geo) on top of fused results.

Want help wiring this into a production retrieval stack? [Join us on Discord](https://qdrant.to/discord).